# Turn repository files into chunks

Tutorial 40 selected repository file names without reading their contents. This tutorial takes the next step: read one real file, recognize its natural functions, and turn them into chunks.

This tutorial stops at chunks. Tutorial 60 sends those chunks to an embedding model.

The first two code cells find QAChat and check its `.venv` environment. Missing packages are repaired with `uv`. If asked, select the printed `.venv` kernel and rerun the cell.

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from setup.notebook import prepare_notebook

root = prepare_notebook(root)


## 1. Read one selected file

Here, **source** means the text inside a selected repository file. We use one real QAChat file so every following result can be inspected.

To inspect a text-based PDF instead, set `SOURCE_PATH` to its path and use `extensions=[".pdf"]` below. Each non-empty PDF page becomes a natural unit; scanned image PDFs need OCR.


In [ ]:
SOURCE_PATH = "qachat/repository.py"


In [ ]:
from qachat.repository import scan_repository

report = scan_repository(
    root,
    extensions=[".py"],
    include_paths=[SOURCE_PATH],
)

print(report["selected_files"])


In [ ]:
from qachat.sources.sources import read_repository_files

source_file = read_repository_files(report)[0]

print(f"Path: {source_file.path}")
print(f"Lines: {len(source_file.text.splitlines())}")


## 2. Find its natural units

A natural unit follows the file's own structure. In this file, QAChat keeps the module-level context and each complete top-level function as separate units. It reads the syntax but never runs the functions.


In [ ]:
from qachat.sources.sources import extract_source_units

units = extract_source_units([source_file])

for unit in units:
    print(unit)

In [ ]:
function_units = [
    unit
    for unit in units
    if unit.kind == "python_function"
]

print(f"Natural units: {len(units)}")
print(f"Function units: {len(function_units)}")


In [ ]:
def count_words(text: str) -> int:
    return len(text.split())


for unit in units:
    location = f"{unit.start_line}-{unit.end_line}"
    size = count_words(unit.text)
    print(f"{location:<9} {size:>3} words ({unit.kind})   {unit.symbol}")


The line ranges come from the real file. Each function unit starts at its `def` line and ends with that complete function.


In [ ]:
example_unit = min(
    function_units,
    key=lambda unit: count_words(unit.text),
)

print(f"Example natural unit: {example_unit.symbol}")


In [ ]:
print(example_unit.text)

## 3. Turn natural units into chunks

A chunk is the piece retrieval will search. A function stays whole when it is below the maximum size. A larger function is divided into overlapping parts, and every part keeps the same function name and source lines.

Smaller chunks are more precise; larger chunks keep more surrounding context. Choose a maximum below the embedding model's input limit, then adjust it by checking retrieval results. These small word values are only used to make the split visible here.


In [ ]:
TARGET_WORDS = 120
MAX_WORDS = 160
OVERLAP_WORDS = 20


In [ ]:
from qachat.chunking import chunk_source_units

chunks = chunk_source_units(
    units,
    count_tokens=count_words,
    target_tokens=TARGET_WORDS,
    max_tokens=MAX_WORDS,
    overlap_tokens=OVERLAP_WORDS,
)

print(f"Natural units: {len(units)}")
print(f"Searchable chunks: {len(chunks)}")


In [ ]:
for chunk in chunks:
    print(chunk)

### A complete function chunk

The shortest real function fits below the maximum, so its natural unit and its chunk are identical.


In [ ]:
whole_unit = min(
    function_units,
    key=lambda unit: count_words(unit.text),
)
whole_chunks = [
    chunk
    for chunk in chunks
    if chunk.symbol == whole_unit.symbol
]

assert len(whole_chunks) == 1
assert whole_chunks[0].text == whole_unit.text

print(f"Function: {whole_unit.symbol}")
print(f"Natural unit: {whole_unit.start_line}-{whole_unit.end_line}")
print(f"Chunk: {whole_chunks[0].start_line}-{whole_chunks[0].end_line}")


### Chunks from one larger function

Now select the smallest function that exceeds the maximum. Its chunks cover only parts of the function. Overlapping line ranges show where a little boundary context was repeated.


In [ ]:
oversized_units = [
    unit
    for unit in function_units
    if count_words(unit.text) > MAX_WORDS
]
split_unit = min(
    oversized_units,
    key=lambda unit: count_words(unit.text),
)
split_chunks = [
    chunk
    for chunk in chunks
    if chunk.symbol == split_unit.symbol
]

assert len(split_chunks) > 1

print(f"Function: {split_unit.symbol}")
print(f"Natural unit: {split_unit.start_line}-{split_unit.end_line}")


In [ ]:
for number, chunk in enumerate(split_chunks, start=1):
    starts_at_def = chunk.text.lstrip().startswith("def ")
    print(
        f"Chunk {number}: lines {chunk.start_line}-{chunk.end_line}, "
        f"{count_words(chunk.text)} words, "
        f"starts at def: {starts_at_def}"
    )


Every chunk still carries the original function symbol, so later search results can point back to the correct function and lines.

## Result and next step

`chunks` is now the final output of this tutorial. Tutorial 60 uses the embedding model's real token counter and converts chunks into vectors. A later retrieval tutorial will compare those vectors.
